In [1]:
import os
import platform
import sys
import time

import numpy as np
import h5py
import pint
import tables

In [2]:
match platform.system():
    case "Linux":
        sys.path.insert(1, os.path.abspath(".."))
        import lysis
        from lysis.util import Q_
        lysis_root = os.path.join("/", "home", "bpaynter", "git", "UCO-OpResearch", "lysis")
    case "Windows":
        import src.python.lysis as lysis

In [3]:
#Need to make considerations for loading the data from an existing HDF5 instead of just loading a new one every time.
#Need to start moving this notebook to lysis\src\python\lysis\util\ and replace datastore.py with this code becoming methods.
#Need to create some sort of class
run_code="2024-04-16-1923"
r = lysis.util.Run(os.path.join(lysis_root, "data"), run_code=run_code)
r.read_file()
r.macro_params.total_molecules

/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:279: RuntimeWarning: Run parameter file does not contain Microscale parameters. Using defaults.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter pore_size has no units. Assuming centimeters.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter diffusion_coeff has no units. Assuming cm^2/s.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter total_time has no units. Assuming seconds.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter save_interval has no units. Assuming sec.
  warnings.warn(


21105

In [4]:
# r.to_file()

Creates a variable f that references the file we want to access.
Need to change variables according to the name of you .h5 file

In [5]:
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'w')

Creates folder and data set structure

In [6]:
runs = h5file.create_group("1-PKd") 
micro_data = runs.create_group("micro_data") 
macro_data = runs.create_group("macro_data")


#Micro data set initializations
pli_first_time = micro_data.create_dataset("pli_first_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
tpa_final_num = micro_data.create_dataset("tpa_final_num", (r.micro_params.simulations, ), dtype= np.uint8, compression="gzip")
fiber_degraded = micro_data.create_dataset("fiber_degraded", (r.micro_params.simulations, ), dtype= np.bool_, compression="gzip")
sim_final_time = micro_data.create_dataset("sim_final_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
pli_generated_num = micro_data.create_dataset("pli_generated_num", (r.micro_params.simulations, ), dtype= np.uint16, compression="gzip")
tpa_leaving_time = micro_data.create_dataset("tpa_leaving_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
tpa_unbound_by_pli = micro_data.create_dataset("tpa_unbound_by_pli", (r.micro_params.simulations, ), dtype= np.bool_, compression="gzip")
tpa_unbound_kinetic = micro_data.create_dataset("tpa_unbound_kinetic", (r.micro_params.simulations, ), dtype = np.bool_, compression="gzip")

#Macro data set initializations

for i in range(0 , r.macro_params.simulations):
    simulation = macro_data.create_group(f"sim_{i:02}")
    fiber_degrade_time = simulation.create_dataset("fiber_degrade_time", (1 , 3) , dtype=np.float64, maxshape = (None, 3), compression="gzip", chunks=(10_000, 3))
    tpa_bind_events = simulation.create_dataset("tpa_bind_events", (1 , 10) , dtype=np.float64 , maxshape = (None, 10), compression="gzip", chunks=(10_000, 10)) #snapshot time = 
    snapshot_time = simulation.create_dataset("snapshot_time", (1,) , dtype=np.float64 , maxshape = (None,), compression="gzip", chunks=(10_000,))
    #tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules , 1) , dtype=np.int32 , maxshape = (r.macro_params.total_molecules, None)) #I uncapped the max number of rows so the data can fit
    tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules, 1) , dtype=np.int32 , maxshape = (r.macro_params.total_molecules, None), compression="gzip", chunks=(r.macro_params.total_molecules, 5))
    tpa_transit_time = simulation.create_dataset("tpa_transit_time", (r.macro_params.total_molecules,) , dtype=np.float64, compression="gzip")
    


Reads in Micro scale data into file system

In [7]:
file_code = "PLG2_tPA01_TB-xiii"

pli_first_time[:] = np.fromfile(
    os.path.join(r.os_path, f"firstPLi_{file_code}.dat"),
)
pli_first_time.attrs["units"] = "seconds"

tpa_final_num[:] = np.fromfile(os.path.join(r.os_path, f"lasttPA_{file_code}.dat"), dtype=np.int32)
tpa_final_num.attrs["units"] = "none"

fiber_degraded[:] = np.fromfile(
    os.path.join(r.os_path, f"lyscomplete_{file_code}.dat"), 
    dtype=np.int32
).astype(bool)
fiber_degraded.attrs["units"] = "none"

sim_final_time[:] = np.fromfile(os.path.join(r.os_path, f"lysis_{file_code}.dat"))
sim_final_time.attrs["units"] = "seconds"

pli_generated_num[:] = np.fromfile(os.path.join(r.os_path, f"PLi_{file_code}.dat"), dtype=np.int32).astype(np.uint16)
pli_generated_num.attrs["units"] = "none"

tpa_leaving_time[:] = np.fromfile(os.path.join(r.os_path, f"tPA_time_{file_code}.dat"))
tpa_leaving_time.attrs["units"] = "seconds"

tpa_unbound_by_pli[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAPLiunbd_{file_code}.dat"), 
    dtype=np.int32
).astype(bool)
tpa_unbound_by_pli.attrs["units"] = "none"

tpa_unbound_kinetic[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAPLiunbd_{file_code}.dat"), 
    dtype=np.int32
).astype(bool)
tpa_unbound_kinetic.attrs["units"] = "none"

Reads in Macro scale Fiber Degrade Time Data

In [8]:

for i in range (0 , 10):
    file_reference = h5file[f"1-PKd/macro_data/sim_0{i}/fiber_degrade_time"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.loadtxt(os.path.join(r.os_path, f"0{i}", f"f_deg_list_{macro_file_code}.dat") , delimiter = ",")
    data = np.reshape(data, (-1, 3))
    data[: , 1] = data[: , 1] - 1   #The data in column 1 is 1 indexed, so we need to convert it to 0 indexed
    file_reference.resize(data.shape)
    file_reference[:] = data
    #Adding Attributes to datasets
    file_reference.attrs["units"] = ["seconds" , "none" , "seconds"]

Reads in Macro scale TPA Bind Events Data | Needs to be reworked for efficiency 51 seconds (5.5 min on Buddy)

I believe what is taking so long is the resizing of the HDF dataset that is going on. Perhaps by specifying size on instatiation, we can avoid this.

In [9]:
for i in range (0 , 10):
    file_reference = h5file[f"1-PKd/macro_data/sim_0{i}/tpa_bind_events"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.loadtxt(os.path.join(r.os_path, f"0{i}", f"m_bind_t_{macro_file_code}.dat") , delimiter = ",")
    data = np.reshape(data, (-1, 4))
    data[:,[1,3]] = data[:,[1,3]] - 1   #The data in column 1 and 3 is 1 indexed, so we need to convert it to 0 indexed
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = ["seconds" , "none" , "none", "none"]
    

Reads in Macro scale snapshot time data

In [10]:
for i in range (0 , 10):
    file_reference = h5file[f"1-PKd/macro_data/sim_0{i}/tpa_location_snapshot"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.fromfile(os.path.join(r.os_path, f"0{i}", f"m_loc_{macro_file_code}.dat") , dtype = np.int32).reshape(r.macro_params.total_molecules, -1)
    data = data - 1 #The data is 1 indexed, so we need to convert it to 0 indexed
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = 'none'

Reads in Macro scale TPA Transit Time Data

In [11]:
for i in range (0 , 10):
    file_reference = h5file[f"1-PKd/macro_data/sim_0{i}/tpa_transit_time"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.fromfile(os.path.join(r.os_path, f"0{i}", f"mfpt_{macro_file_code}.dat") , dtype = np.float64) #.reshape(-1,1)
    file_reference[:] = data
    file_reference.attrs["units"] = "seconds"

Reads in Macro scale Snapshot Time Data

In [12]:
for i in range (0 , 10):
    file_reference = h5file[f"1-PKd/macro_data/sim_0{i}/snapshot_time"]
    macro_file_code = f"TB-xiii__21_105_0{i}"
    data = np.fromfile(os.path.join(r.os_path, f"0{i}", f"tsave_{macro_file_code}.dat") , dtype = np.float64) #.reshape(-1,1)
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = "seconds"

In [13]:
h5file.close()

Post Processing

In [14]:
import pint
u = pint.UnitRegistry()
Q = u.Quantity

In [14]:
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'r')

In [22]:
H5_dataset = h5file["1-PKd/macro_data/sim_00/fiber_degrade_time"]
unit_array = H5_dataset.attrs["units"]
dataset = np.array(h5file["1-PKd/macro_data/sim_00/fiber_degrade_time"])
event_time = dataset[:,0]
legs2 = [400.0, 300.0] * u.centimeter
legs2 = event_time * u(unit_array[0])
print(legs2.to('min'))
#print(legs2)
unit_array

[0.00031684 0.00031684 0.00126736 ... 47.73796595999999 48.082054199999995 48.75819075999999] minute


array(['seconds', 'dimensionless', 'seconds'], dtype=object)

In [16]:
data = h5file["1-PKd/micro_data/tpa_final_num"]
data.attrs["units"]

'none'

In [16]:
r.macro_params

MacroParameters(pore_size=<Quantity(0.000534, 'centimeter')>, diffusion_coeff=<Quantity(5e-07, 'centimeter ** 2 / second')>, forced_unbind=0.074101, average_bound_time=<Quantity(27.8, 'second')>, cols=19, rows=184, fiber_rows=72, empty_rows=112, empty_edges=6272, full_row=56, xz_row=37, total_edges=10285, total_fibers=4013, total_molecules=21105, moving_probability=0.2, simulations=10, total_time=<Quantity(0, 'second')>, time_step=<Quantity(0.0095052, 'second')>, total_time_steps=0, seed=3134955532, state=(129281, 362436069, 123456789, 3134955532), input_data=['unbinding_time', 'lysis_time_dist', 'total_lyses'], output_data=['degradation_state', 'molecule_location', 'molecule_state', 'save_time'], save_interval=<Quantity(10, 'second')>, number_of_saves=1, macro_version='macro_diffuse_into_and_along__external', log_lvl=30, duplicate_fortran=False, processing_library='numpy')